# Train/Test Split and Exploratory Data Analysis

## Introduction

This notebook prepares the cleaned welding dataset for the machine learning stage. The first objective is to create a train/test split while avoiding data leakage, especially when several rows may correspond to the same physical weld. After the split, the training set will be used for exploratory data analysis, including correlations, PCA, and feature selection. The test set will remain untouched until the final evaluation of the models.

In [1]:
import pandas as pd

df = pd.read_csv("../data/welddb_cleaned_units.csv")

df.head()

,C,Si,Mn,S,P,Ni,Cr,Mo,V,O_ppm,...,charpy_toughness_J,weld_id,weld_type_FCA,weld_type_GMAA,weld_type_GTAA,weld_type_MMA,weld_type_SA,weld_type_SAA,weld_type_TSA,AC_DC_unknown
0,0.037,0.30,0.65,0.008,0.012,0.0,0.0,0.0,0.0,423.0,...,NaN,Evans-Ni/CMn-1990/1991-0Aaw,0,0,0,1,0,0,0,0
1,0.037,0.30,0.65,0.008,0.012,0.0,0.0,0.0,0.0,423.0,...,100.0,Evans-Ni/CMn-1990/1991-0Aawch,0,0,0,1,0,0,0,0
2,0.037,0.30,0.65,0.008,0.012,0.0,0.0,0.0,0.0,423.0,...,100.0,Evans-Ni/CMn-1990/1991-0Aht,0,0,0,1,0,0,0,0
3,0.037,0.31,1.03,0.007,0.014,0.0,0.0,0.0,0.0,423.0,...,NaN,Evans-Ni/CMn-1990/1991-0Baw,0,0,0,1,0,0,0,0
4,0.037,0.31,1.03,0.007,0.014,0.0,0.0,0.0,0.0,423.0,...,100.0,Evans-Ni/CMn-1990/1991-0Bawch,0,0,0,1,0,0,0,0


## 2. Inspect Weld Identifiers

Before splitting the dataset, we need to check how the weld identifiers are structured. Some rows may represent different measurements of the same physical weld. These rows must stay together in either the training set or the test set to avoid data leakage.

In [2]:
print("Number of rows:", len(df))
print("Number of unique weld IDs:", df["weld_id"].nunique())

df["weld_id"].value_counts().head(30)

Number of rows: 1639
Number of unique weld IDs: 1477


weld_id
Pat-1981-RE2L/BX200         5
Pat-1981-S2/BX200           5
Pat-1981-RE3L/BX200         5
Pat-1981-WB3/BX200          5
Pat-1981-S4/BX200           5
Pat-1981-S6/BX200           5
Pat-1981-S2Mo/BX200         5
Pat-1981-S4Mo/BX200         5
Pat-1981-RE2L/BX400         5
Pat-1981-S2/BX400           5
Pat-1981-RE3L/BX400         5
Pat-1981-WB3/BX400          5
Pat-1981-S4/BX400           5
Pat-1981-S6/BX400           5
Pat-1981-S2Mo/BX400         5
Evans-B/9Cr1Mo-1994-CM57    4
Evans-B/9Cr1Mo-1994-CM58    4
Evans-B/9Cr1Mo-1994-CM59    4
Evans-B/9Cr1Mo-1994-CM60    4
Evans-B/9Cr1Mo-1994-CM61    4
Evans-B/9Cr1Mo-1994-CM62    4
Evans-B/9Cr1Mo-1994-CM63    4
Evans-B/9Cr1Mo-1994-CM64    4
Pat-1981-S4Mo/BX400         4
RC81033                     4
Evans-AlTi-1994-80/8ht      3
Evans-AlTi-1994-80/47ht     3
Evans-AlTi-1994-80/160ht    3
Evans-AlTi-1994-80/280ht    3
Evans-AlTi-1994-80/480ht    3
Name: count, dtype: int64

In [3]:
df[["weld_id"]].head(50)

,weld_id
0,Evans-Ni/CMn-1990/1991-0Aaw
1,Evans-Ni/CMn-1990/1991-0Aawch
2,Evans-Ni/CMn-1990/1991-0Aht
3,Evans-Ni/CMn-1990/1991-0Baw
4,Evans-Ni/CMn-1990/1991-0Bawch
5,Evans-Ni/CMn-1990/1991-0Bht
6,Evans-Ni/CMn-1990/1991-0Caw
7,Evans-Ni/CMn-1990/1991-0Cawch
8,Evans-Ni/CMn-1990/1991-0Cht
9,Evans-Ni/CMn-1990/1991-0Daw


### 2.1 Investigate related weld IDs

Some weld identifiers share the same base name but have different suffixes such as `aw`, `awch`, `ht`, or `mic`. Before creating the train/test split, we need to determine whether these rows correspond to the same underlying weld or to genuinely different weld conditions.

In [4]:
cols_to_compare = [
    "weld_id",
    "C", "Si", "Mn", "Ni", "Cr", "Mo",
    "current_A", "voltage_V",
    "heat_input_kJ_mm",
    "interpass_temp_C",
    "pwht_temp_C",
    "pwht_time_h"
]

df[df["weld_id"].str.contains("Evans-Ni/CMn-1990/1991-0A", na=False)][cols_to_compare]

,weld_id,C,Si,Mn,Ni,Cr,Mo,current_A,voltage_V,heat_input_kJ_mm,interpass_temp_C,pwht_temp_C,pwht_time_h
0,Evans-Ni/CMn-1990/1991-0Aaw,0.037,0.3,0.65,0.0,0.0,0.0,170.0,21.0,1.0,200.0,250.0,14.0
1,Evans-Ni/CMn-1990/1991-0Aawch,0.037,0.3,0.65,0.0,0.0,0.0,170.0,21.0,1.0,200.0,0.0,0.0
2,Evans-Ni/CMn-1990/1991-0Aht,0.037,0.3,0.65,0.0,0.0,0.0,170.0,21.0,1.0,200.0,580.0,2.0


In [5]:
examples = [
    "Evans-Ni/CMn-1990/1991-0A",
    "Pat-1981-WB3/BX200",
    "Evans-B/9Cr1Mo-1994-CM63"
]

for ex in examples:
    print("\n###", ex)
    display(df[df["weld_id"].str.contains(ex, na=False)][[
        "weld_id",
        "C", "Si", "Mn", "Ni", "Cr", "Mo",
        "current_A", "voltage_V",
        "heat_input_kJ_mm",
        "interpass_temp_C",
        "pwht_temp_C",
        "pwht_time_h"
    ]])


### Evans-Ni/CMn-1990/1991-0A


,weld_id,C,Si,Mn,Ni,Cr,Mo,current_A,voltage_V,heat_input_kJ_mm,interpass_temp_C,pwht_temp_C,pwht_time_h
0,Evans-Ni/CMn-1990/1991-0Aaw,0.037,0.3,0.65,0.0,0.0,0.0,170.0,21.0,1.0,200.0,250.0,14.0
1,Evans-Ni/CMn-1990/1991-0Aawch,0.037,0.3,0.65,0.0,0.0,0.0,170.0,21.0,1.0,200.0,0.0,0.0
2,Evans-Ni/CMn-1990/1991-0Aht,0.037,0.3,0.65,0.0,0.0,0.0,170.0,21.0,1.0,200.0,580.0,2.0



### Pat-1981-WB3/BX200


,weld_id,C,Si,Mn,Ni,Cr,Mo,current_A,voltage_V,heat_input_kJ_mm,interpass_temp_C,pwht_temp_C,pwht_time_h
960,Pat-1981-WB3/BX200,0.08,0.31,1.67,0.0,0.0,0.03,625.0,75.36,4.6,200.0,0.0,0.0
961,Pat-1981-WB3/BX200,0.08,0.31,1.67,0.0,0.0,0.03,625.0,75.36,4.6,200.0,0.0,0.0
962,Pat-1981-WB3/BX200,0.08,0.31,1.67,0.0,0.0,0.03,625.0,75.36,4.6,200.0,0.0,0.0
963,Pat-1981-WB3/BX200,0.08,0.31,1.67,0.0,0.0,0.03,625.0,75.36,4.6,200.0,0.0,0.0
964,Pat-1981-WB3/BX200,0.08,0.31,1.67,0.0,0.0,0.03,625.0,75.36,4.6,200.0,0.0,0.0



### Evans-B/9Cr1Mo-1994-CM63


,weld_id,C,Si,Mn,Ni,Cr,Mo,current_A,voltage_V,heat_input_kJ_mm,interpass_temp_C,pwht_temp_C,pwht_time_h
208,Evans-B/9Cr1Mo-1994-CM63,0.086,0.2,1.04,0.0,8.98,1.01,180.0,34.0,1.0,175.0,750.0,8.0
209,Evans-B/9Cr1Mo-1994-CM63,0.086,0.2,1.04,0.0,8.98,1.01,180.0,34.0,1.0,175.0,750.0,8.0
210,Evans-B/9Cr1Mo-1994-CM63,0.086,0.2,1.04,0.0,8.98,1.01,180.0,34.0,1.0,175.0,750.0,8.0
211,Evans-B/9Cr1Mo-1994-CM63,0.086,0.2,1.04,0.0,8.98,1.01,180.0,34.0,1.0,175.0,750.0,8.0


## 3. Train/Test Split

To avoid direct data leakage, all rows of the same weld deposit are kept in the same split. The examples above show that one deposit is spread over several `weld_id`: `0Aaw` holds the tensile result, `0Aawch` the as-welded Charpy result and `0Aht` the same deposit after a heat treatment. Splitting on `weld_id` would put `0Aaw` in train and `0Aawch` in test, so the split is done on a new `weld_group` column instead. Different post-weld treatment conditions are considered distinct observations because treatment temperature and duration are input features of the model.

- **Rows without any target:** The 105 rows with no tensile and no Charpy result are dropped, they cannot be used for supervised learning.
- **weld_group:** One id per weld deposit, two rows are in the same group when they have the same chemistry and the same welding parameters. `pwht_temp_C`, `pwht_time_h` and `charpy_temp_C` are left out on purpose, so that the as-welded rows, the heat treated rows and every Charpy row of one deposit end up in the same group.
- **Split:** 80% train / 20% test by `weld_group`, with `StratifiedGroupKFold` (5 folds, the first fold is the test set). It is stratified on the targets available in each row (1 = tensile only, 2 = Charpy only, 3 = both), so that both sets keep the same share of tensile and Charpy rows.

In [6]:
from sklearn.model_selection import StratifiedGroupKFold

tensile_cols = ["yield_strength_MPa", "uts_MPa", "elongation_pct", "reduction_area_pct"]
target_cols = tensile_cols + ["charpy_toughness_J"]

df = df[df[target_cols].notna().any(axis=1)].reset_index(drop=True)

chem_cols = ["C", "Si", "Mn", "S", "P", "Ni", "Cr", "Mo", "V", "O_ppm", "Ti_ppm", "N_ppm", "Al_ppm", "Nb_ppm"]
process_cols = ["current_A", "voltage_V", "AC_DC", "electrode_polarity", "AC_DC_unknown", "heat_input_kJ_mm", "interpass_temp_C"]
process_cols += [c for c in df.columns if c.startswith("weld_type_")]

df["weld_group"] = df.groupby(chem_cols + process_cols).ngroup()

has_tensile = df[tensile_cols].notna().any(axis=1)
has_charpy = df["charpy_toughness_J"].notna()
target_profile = has_tensile.astype(int) + 2 * has_charpy.astype(int)

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, target_profile, groups=df["weld_group"])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Rows with at least one target:", len(df))
print("Number of weld groups:", df["weld_group"].nunique())
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

Rows with at least one target: 1534
Number of weld groups: 602
Train rows: 1229
Test rows: 305


### 3.1 Leakage Check

The check is done on `weld_group`, the unit of the split. Two `weld_id` (`Stil-TiB-1978-1.5Mn/OP121TT` and `Stil-TiB-1978-1.5Mn/OP41TT`) appear in both sets, but with a different chemistry: they are two different deposits sharing the same name.

In [7]:
overlap = set(train_df["weld_group"]) & set(test_df["weld_group"])

print("Number of weld groups present in both train and test:", len(overlap))

Number of weld groups present in both train and test: 0


In [8]:
print("Train percentage:", round(len(train_df) / len(df) * 100, 2), "%")
print("Test percentage:", round(len(test_df) / len(df) * 100, 2), "%")

print("Unique weld groups in train:", train_df["weld_group"].nunique())
print("Unique weld groups in test:", test_df["weld_group"].nunique())

Train percentage: 80.12 %
Test percentage: 19.88 %
Unique weld groups in train: 483
Unique weld groups in test: 119


### 3.2 Save the Split

`weld_group` is kept in both files: the cross-validation on the train set must also be grouped on it, for the same reason as this split. From now on the test set is not opened until the final evaluation.

In [9]:
train_df.to_csv("../data/train.csv", index=False)
test_df.to_csv("../data/test.csv", index=False)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (1229, 38)
Test shape: (305, 38)


## 4. Define Potential Target Variables

Before performing PCA or feature selection, we need to decide which variables represent weld quality. Mechanical properties such as yield strength, tensile strength, elongation, Charpy toughness, hardness, and related measurements are possible target variables. These variables should not be included as input features if they are used to define or predict weld quality.

In [10]:
train_df.columns.tolist()

['C',
 'Si',
 'Mn',
 'S',
 'P',
 'Ni',
 'Cr',
 'Mo',
 'V',
 'O_ppm',
 'Ti_ppm',
 'N_ppm',
 'Al_ppm',
 'Nb_ppm',
 'current_A',
 'voltage_V',
 'AC_DC',
 'electrode_polarity',
 'heat_input_kJ_mm',
 'interpass_temp_C',
 'pwht_temp_C',
 'pwht_time_h',
 'yield_strength_MPa',
 'uts_MPa',
 'elongation_pct',
 'reduction_area_pct',
 'charpy_temp_C',
 'charpy_toughness_J',
 'weld_id',
 'weld_type_FCA',
 'weld_type_GMAA',
 'weld_type_GTAA',
 'weld_type_MMA',
 'weld_type_SA',
 'weld_type_SAA',
 'weld_type_TSA',
 'AC_DC_unknown',
 'weld_group']

In [11]:
target_candidates = [
    "yield_strength_MPa",
    "uts_MPa",
    "elongation_pct",
    "reduction_area_pct",
    "charpy_temp_C",
    "charpy_toughness_J"
]

target_summary = pd.DataFrame({
    "non_missing": train_df[target_candidates].notna().sum(),
    "missing": train_df[target_candidates].isna().sum(),
    "missing_pct": train_df[target_candidates].isna().mean() * 100
})

target_summary

,non_missing,missing,missing_pct
yield_strength_MPa,612,617,50.203417
uts_MPa,591,638,51.912124
elongation_pct,549,680,55.329536
reduction_area_pct,552,677,55.085435
charpy_temp_C,704,525,42.717657
charpy_toughness_J,704,525,42.717657


## 5. Training Set Overview

Before selecting the final target and features, we examine the general structure of the training data. This includes dataset size, missing values, and the distribution of the main input variables.

In [12]:
print("Training set shape:", train_df.shape)

missing_train = pd.DataFrame({
    "missing_count": train_df.isna().sum(),
    "missing_pct": train_df.isna().mean() * 100
}).sort_values("missing_pct", ascending=False)

missing_train

Training set shape: (1229, 38)


,missing_count,missing_pct
elongation_pct,680,55.329536
reduction_area_pct,677,55.085435
uts_MPa,638,51.912124
yield_strength_MPa,617,50.203417
charpy_temp_C,525,42.717657
charpy_toughness_J,525,42.717657
Ni,0,0.000000
P,0,0.000000
S,0,0.000000
Mn,0,0.000000


### Observation

All input features are complete in the training set after preprocessing. Missing values are only present in the mechanical-property variables that may be used as prediction targets. Therefore, no additional imputation is required for the input features at this stage.